# LAB 6 (Bonus) — Agent + NVIDIA-Verified Skill 🔐
เชื่อม **NVIDIA Agent Skills** (คลังความสามารถ) + **agent** (Nemotron) + **signing** (ความปลอดภัย — โยง LAB 5)

**skill = ชุดคำสั่งสอน agent ใช้ซอฟต์แวร์ NVIDIA ให้ถูก** (cuDF, cuOpt, ...) ที่ NVIDIA **เซ็นรับรอง** — กด `Shift+Enter` ทีละเซลล์

In [ ]:
import os
from openai import OpenAI
client = OpenAI(base_url=os.environ['OPENAI_BASE_URL'], api_key='EMPTY'); MODEL=os.environ['MODEL']
SKILLS='/home/jovyan/nvidia-skills'
SKILL='accelerated-computing-cudf'   # ลองเปลี่ยนเป็น cuopt-numerical-optimization-api-python ได้
import os.path as p; base=p.join(SKILLS,'skills',SKILL)
print('skill:', SKILL)

In [ ]:
# 1) บัตรประจำตัว skill (skill-card) — ทำอะไร / เจ้าของ / ความเสี่ยง
print(open(p.join(base,'skill-card.md')).read()[:900])

In [ ]:
# 2) ✅ ตรวจว่า 'เซ็นรับรองโดย NVIDIA' — มีลายเซ็น OMS + root cert (integrity + provenance)
sig=p.join(base,'skill.oms.sig'); cert=p.join(SKILLS,'nv-agent-root-cert.pem')
print('OMS signature:', '✅ พบ ('+str(os.path.getsize(sig))+' bytes)' if os.path.exists(sig) else '❌ ไม่มี')
print('NVIDIA root cert:', '✅ พบ' if os.path.exists(cert) else '❌ ไม่มี')
print('\n→ เซ็นแบบ OMS (OpenSSF Model Signing) ครอบทั้งโฟลเดอร์ skill = ยืนยันว่าไม่ถูกแก้ + มาจาก NVIDIA จริง')
print('  (เทียบ LAB 5: deny-by-default — ของที่ยังไม่ verify อย่าเพิ่งให้ agent ใช้)')

In [ ]:
# 3) ถาม agent 'ก่อน' ใส่ skill (ความรู้ทั่วไป)
Q='ฉันมี pandas groupby บนข้อมูล 5 ล้านแถว ช้ามาก จะเร่งบน GPU ของ NVIDIA ยังไง? ตอบสั้นๆ เป็นขั้น'
r=client.chat.completions.create(model=MODEL,max_tokens=200,messages=[{'role':'user','content':Q}])
print('🟢 ไม่มี skill:\n', r.choices[0].message.content)

In [ ]:
# 4) ใส่ SKILL.md เป็น 'คำสั่งประจำตัว' ของ agent แล้วถามใหม่ → ตอบตามแนวทาง NVIDIA จริง
skill_md=open(p.join(base,'SKILL.md')).read()[:3500]   # ตัดให้พอดี context
r=client.chat.completions.create(model=MODEL,max_tokens=250,messages=[{'role':'system','content':skill_md},{'role':'user','content':Q}])
print('🔵 มี skill (cuDF):\n', r.choices[0].message.content)

**บทเรียน (เชื่อม 3 อย่าง):**
- **Skill** = ความสามารถพกพาได้ (สอน agent ใช้ cuDF/cuOpt ให้ถูก) → คำตอบดีขึ้น/เฉพาะทางขึ้น
- **Signed/Verified** = เชื่อถือได้ (เซ็น OMS โดย NVIDIA) → ติดตั้งเฉพาะที่ verify แล้ว (เหมือน app store) ↔ ความปลอดภัย LAB 5
- **NemoClaw** = production stack ที่รวม **agent + OpenShell sandbox + skills** เข้าด้วยกัน (ดูผู้สอนสาธิต)

> ติดตั้ง skill จริงในเครื่องตัวเอง: `npx skills add nvidia/skills` (เลือก skill) — แล้ว agent โหลด skill เมื่อเจองานที่เกี่ยว